# Feature Encoding in Machine Learning

This notebook explains Feature Encoding concepts with theory and practical Python examples.

## What is Feature Encoding?

Feature Encoding converts categorical data into numerical form so Machine Learning algorithms can process it.

In [1]:
import pandas as pd

df = pd.DataFrame({
    'City': ['Mumbai', 'Delhi', 'Pune', 'Delhi'],
    'Education': ['High School', 'Bachelor', 'Master', 'Bachelor'],
    'Salary': [50000, 60000, 70000, 65000]
})

df

,City,Education,Salary
0,Mumbai,High School,50000
1,Delhi,Bachelor,60000
2,Pune,Master,70000
3,Delhi,Bachelor,65000


## 1. Label Encoding

Assigns a unique integer to each category.

**Use for:** Tree-based models like Random Forest and XGBoost.

❌ Not good for Linear Regression / Logistic Regression
(because model assumes order)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['City_label'] = le.fit_transform(df['City'])  # or add map like Delhi -> 0, Mum ->1 etc with new column

print(le.classes_)  # order of labeling

df

['Delhi' 'Mumbai' 'Pune']


,City,Education,Salary,City_label
0,Mumbai,High School,50000,1
1,Delhi,Bachelor,60000,0
2,Pune,Master,70000,2
3,Delhi,Bachelor,65000,0


## 2. One-Hot Encoding

Creates a new column for each category.

**Use for:** Nominal categorical features (no order).

⚠ Problem

High Cardinality Issue

Example:

User_ID → 50,000 values

➡ creates 50,000 columns ❌

In [6]:
pd.get_dummies(df['City'])

,Delhi,Mumbai,Pune
0,False,True,False
1,True,False,False
2,False,False,True
3,True,False,False


In [10]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df[['City']])

print(encoded)

[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


## 3. Ordinal Encoding

Used when categories have natural ordering.

Example: High School < Bachelor < Master

✅ Use Cases

Rating

Education level

Experience level

In [11]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(
    categories=[['High School', 'Bachelor', 'Master']]
)

df['Education_ordinal'] = encoder.fit_transform(df[['Education']])

df

,City,Education,Salary,City_label,Education_ordinal
0,Mumbai,High School,50000,1,0.0
1,Delhi,Bachelor,60000,0,1.0
2,Pune,Master,70000,2,2.0
3,Delhi,Bachelor,65000,0,1.0


## 4. Target Encoding

Replaces categories with the mean value of the target variable.

below examples shows - City influences salary level

✅ Best For

✔ High-cardinality columns
✔ Kaggle-level ML models

In [22]:
# Install first if needed
# !pip install category_encoders

import category_encoders as ce

encoder = ce.TargetEncoder(cols=['City'])
df_target = encoder.fit_transform(df[['City']], df['Salary'])

df_target

,City
0,59786.279663
1,61427.313831
2,62388.449151
3,61427.313831


In [ ]:
mean = df.groupby(['City'])['Salary'].mean()

df['City_Target_enc'] = df['City'].map(mean) # City influences salary level
df

,City,Education,Salary,City_label,Education_ordinal,City_freq,ED_Target_enc,City_Target_enc
0,Mumbai,High School,50000,1,0.0,1,1.0,50000.0
1,Delhi,Bachelor,60000,0,1.0,2,0.0,62500.0
2,Pune,Master,70000,2,2.0,1,2.0,70000.0
3,Delhi,Bachelor,65000,0,1.0,2,0.0,62500.0


## 5. Frequency / Count Encoding

Replaces categories with occurrence frequency.

✅ Useful For

Large datasets

Memory optimization

Production ML pipelines

In [17]:
freq = df['City'].value_counts()

df['City_freq'] = df['City'].map(freq)

df

,City,Education,Salary,City_label,Education_ordinal,City_freq,ED_Target_enc
0,Mumbai,High School,50000,1,0.0,1,1.0
1,Delhi,Bachelor,60000,0,1.0,2,0.0
2,Pune,Master,70000,2,2.0,1,2.0
3,Delhi,Bachelor,65000,0,1.0,2,0.0


## 6. Binary Encoding

Combines Label Encoding and Binary representation to reduce dimensionality.

Binary Encoding reduces dimensionality by converting category indices into binary digits instead of creating one column per category.


Label Encoding + Binary Conversion


✅ Binary Encoding Solution

Instead of 50,000 columns:

𝑙
𝑜
𝑔
2
(
50000
)
≈
16
log
2
	​

(50000)≈16

Only 16 columns needed ✅

Massive dimensionality reduction.


✅ Used When

High cardinal categorical features exist.

In [ ]:
# !pip install category_encoders
import category_encoders as ce

encoder = ce.BinaryEncoder(cols=['City'])
df_binary = encoder.fit_transform(df[['City']])

df_binary  # convert city in to binary represent 0 -> 0 1,  1-> 1 0, 2 -> 1 1

,City_0,City_1
0,0,1
1,1,0
2,1,1
3,1,0


## 7. Embedding Encoding (Deep Learning)

Used in deep learning models to convert categories into dense vectors.

Used in:

NLP

Recommendation Systems

Deep Learning models

Categories mapped into dense vectors.

Example:

Mumbai → [0.21, -0.9, 1.2]

Implemented using:

TensorFlow

PyTorch Embedding Layer

In [24]:
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=10,
    embedding_dim=3
)

embedding.weight

Parameter containing:
tensor([[ 0.6930, -1.1777,  0.1238],
        [ 1.0146,  1.1982,  0.6671],
        [-1.4248,  0.7249, -1.4234],
        [-1.5263,  1.2359,  1.1090],
        [ 1.1949,  1.6522,  1.3970],
        [ 0.4645,  0.6336, -1.0659],
        [-0.6273,  1.5412,  0.0173],
        [ 0.1005,  0.0453, -0.1340],
        [ 0.5119, -0.4275,  0.8609],
        [-0.1596, -0.9404,  0.4422]], requires_grad=True)

## ML PIPELINE

Raw Data
 → Missing Handling
 → Feature Encoding
 → Feature Scaling
 → Feature Store
 → Model Training

## Encoding Selection Cheat Sheet

| Scenario | Encoding |
|-----------|----------|
| Nominal Category | One-Hot |
| Ordered Category | Ordinal |
| Tree Models | Label |
| High Cardinality | Target / Binary |
| Large Dataset | Frequency |
| Deep Learning | Embeddings |